# Stage 02: Deep Dvorak Intensity Estimation (ConvNeXt-V2)
### Multi-Spectral Deep Convolutional Regression & Automated IMD Classification
**Framework:** PyTorch, ConvNeXt-V2, NOAA IBTrACS, Matplotlib
**Operational Target:** Wind Speed MAE < 7.5 knots (< 0.5 Dvorak T-number step)

---
### Physical Overview
The **Vernon Dvorak Technique** (1975, 1984) correlates tropical cyclone intensity with thermal contrast between the warm eye core and the freezing thunderstorm cloud tops ($\Delta T = T_{eye} - T_{eyewall}$). This notebook trains an anchor-stabilized ConvNeXt-V2 network with a **Physics-Informed Loss** function to automate objective intensity estimation.

In [ ]:
import os
import math
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | PyTorch: {torch.__version__}')


## 1. Load NOAA IBTrACS Ground Truth Dataset
We ingest official North Indian Ocean fixes with maximum sustained winds (knots), central pressure (hPa), and storm seasons.

In [ ]:
csv_candidates = [
    '../ai_engine/data/raw_ibtracs/ibtracs.NI.list.v04r00.csv',
    'ai_engine/data/raw_ibtracs/ibtracs.NI.list.v04r00.csv',
    'ibtracs.NI.list.v04r00.csv'
]
csv_path = None
for c in csv_candidates:
    if os.path.exists(c):
        csv_path = c
        break

if csv_path:
    df = pd.read_csv(csv_path, low_memory=False, skiprows=[1])
    df['WIND'] = pd.to_numeric(df['USA_WIND'], errors='coerce').fillna(
        pd.to_numeric(df['WMO_WIND'], errors='coerce')
    )
    df = df.dropna(subset=['WIND']).query('WIND >= 17').copy()
    df['SEASON'] = pd.to_numeric(df['SEASON'], errors='coerce')
    print(f'Valid Cyclonic Observations: {len(df)} rows')
else:
    print('Using synthetic meteorological dataset for standalone execution.')
    df = pd.DataFrame({
        'WIND': np.random.uniform(25, 140, 1000),
        'SEASON': np.random.randint(2000, 2024, 1000)
    })


## 2. Multi-Spectral Radiance Simulation & PyTorch Dataset
Simulate eye-centered 256x256 multi-spectral radiance patches across varying storm intensities and thermal gradients.

In [ ]:
class DeepDvorakDataset(Dataset):
    def __init__(self, winds, num_samples=500, is_train=True):
        self.winds = np.array(winds, dtype=np.float32)
        self.num_samples = num_samples
        self.is_train = is_train

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        target_wind = float(self.winds[idx % len(self.winds)])
        delta_t = 20.0 + (target_wind / 140.0) * 75.0
        eye_temp = 286.0 - np.random.uniform(0, 10)
        eyewall_temp = eye_temp - delta_t

        yy, xx = np.mgrid[0:256, 0:256]
        dist = np.sqrt((xx - 128)**2 + (yy - 128)**2)
        rad = np.ones((256, 256), dtype=np.float32) * 260.0
        rad[dist < 18] = eye_temp
        ring_mask = (dist >= 18) & (dist < 60)
        rad[ring_mask] = eyewall_temp + (dist[ring_mask] - 18) * 0.4
        outer_mask = dist >= 60
        rad[outer_mask] = 230.0 + np.sin(dist[outer_mask] / 12.0) * 15.0
        rad += np.random.normal(0, 2.0, rad.shape)
        norm_rad = np.clip((rad - 180.0) / 120.0, 0.0, 1.0).astype(np.float32)

        if self.is_train and np.random.rand() > 0.5:
            norm_rad = np.fliplr(norm_rad).copy()

        p_hpa = 1010.0 - (target_wind / 6.7)**(1.0 / 0.644)

        return {
            'image': torch.from_numpy(norm_rad).unsqueeze(0),
            'wind': torch.tensor(target_wind, dtype=torch.float32),
            'pressure': torch.tensor(p_hpa, dtype=torch.float32),
            'delta_t': torch.tensor(delta_t, dtype=torch.float32)
        }

train_winds = df[df['SEASON'] <= 2018]['WIND'].values
test_winds  = df[df['SEASON'] >= 2019]['WIND'].values
if len(test_winds) == 0:
    test_winds = train_winds[:100]

train_ds = DeepDvorakDataset(train_winds, num_samples=400, is_train=True)
test_ds  = DeepDvorakDataset(test_winds,  num_samples=100, is_train=False)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False)
print(f'DataLoaders: {len(train_loader)} train batches, {len(test_loader)} test batches')


## 3. ConvNeXt-V2 Model & Physics-Informed Dvorak Loss

In [ ]:
class ConvNeXtBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.BatchNorm2d(dim)
        self.pwconv1 = nn.Conv2d(dim, 4 * dim, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(4 * dim, dim, kernel_size=1)

    def forward(self, x):
        return x + self.pwconv2(self.act(self.pwconv1(self.norm(self.dwconv(x)))))

class DeepDvorakConvNeXt(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=4, stride=4),
            nn.BatchNorm2d(64)
        )
        self.stage1 = nn.Sequential(ConvNeXtBlock(64), ConvNeXtBlock(64))
        self.down1 = nn.Sequential(nn.BatchNorm2d(64), nn.Conv2d(64, 128, kernel_size=2, stride=2))
        self.stage2 = nn.Sequential(ConvNeXtBlock(128), ConvNeXtBlock(128))
        self.down2 = nn.Sequential(nn.BatchNorm2d(128), nn.Conv2d(128, 256, kernel_size=2, stride=2))
        self.stage3 = nn.Sequential(ConvNeXtBlock(256), ConvNeXtBlock(256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head_wind = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.down1(self.stage1(x))
        x = self.down2(self.stage2(x))
        x = self.stage3(x)
        feat = torch.flatten(self.pool(x), 1)
        return self.head_wind(feat).squeeze(-1)

model = DeepDvorakConvNeXt().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
print('ConvNeXt-V2 Initialized successfully.')


## 4. Train with Physics-Informed Dvorak Loss

In [ ]:
epochs = 4
print(f'Training ConvNeXt-V2 for {epochs} epochs...')
for epoch in range(1, epochs + 1):
    model.train()
    total_loss, total_mae = 0.0, 0.0
    for batch in train_loader:
        imgs = batch['image'].to(device)
        winds = batch['wind'].to(device)
        preds = model(imgs)
        mse_loss = F.mse_loss(preds, winds)
        phys_penalty = torch.relu(preds - 165.0).mean() + torch.relu(15.0 - preds).mean()
        loss = mse_loss + 0.2 * phys_penalty

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_mae += F.l1_loss(preds, winds).item()

    mae = total_mae / len(train_loader)
    print(f'Epoch {epoch}/{epochs} | Loss: {total_loss/len(train_loader):.2f} | MAE: {mae:.2f} kts')


## 5. Evaluation & 4-Panel Meteorological Performance Audit

In [ ]:
model.eval()
all_preds, all_gts = [], []
with torch.no_grad():
    for batch in test_loader:
        imgs = batch['image'].to(device)
        winds = batch['wind'].to(device)
        preds = model(imgs)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_gts.extend(winds.cpu().numpy().tolist())

all_preds = np.array(all_preds)
all_gts = np.array(all_gts)
mae = np.mean(np.abs(all_preds - all_gts))
rmse = np.sqrt(np.mean((all_preds - all_gts)**2))
print('=== EVALUATION BENCHMARK ===')
print(f'Validation MAE:  {mae:.2f} knots (Target: < 7.5 kts)')
print(f'Validation RMSE: {rmse:.2f} knots')

# 4-Panel Diagnostic Figure
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

# Panel 1: Parity Plot
axes[0].scatter(all_gts, all_preds, color='#38bdf8', edgecolor='#0284c7', alpha=0.7, s=40)
axes[0].plot([20, 150], [20, 150], 'r--', label='Exact Agreement')
axes[0].fill_between([20, 150], [12.5, 142.5], [27.5, 157.5], color='green', alpha=0.15, label='+-7.5 kts IMD Margin')
axes[0].set_title(f'Panel A: Wind Speed Parity\nMAE = {mae:.2f} kts', fontweight='bold')
axes[0].set_xlabel('IBTrACS Ground Truth (kts)')
axes[0].set_ylabel('ConvNeXt-V2 Estimated (kts)')
axes[0].legend(loc='upper left', fontsize=9)
axes[0].grid(True, linestyle=':', alpha=0.5)

# Panel B: Residuals
residuals = all_preds - all_gts
axes[1].hist(residuals, bins=15, color='#10b981', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Panel B: Intensity Residual Errors', fontweight='bold')
axes[1].set_xlabel('Residual (Predicted - Ground Truth kts)')
axes[1].set_ylabel('Fix Frequency')
axes[1].grid(True, linestyle=':', alpha=0.5)

# Panel C: Dvorak T-number vs Wind Speed
t_nums = np.linspace(1.0, 8.0, 50)
winds_dvorak = 1.84 * (t_nums ** 1.75) * 6.5
axes[2].plot(t_nums, winds_dvorak, color='#f59e0b', linewidth=2.5, label='Dvorak Wind Curve')
axes[2].set_title('Panel C: Vernon Dvorak T-Number Curve', fontweight='bold')
axes[2].set_xlabel('Dvorak T-Number (T1.0 - T8.0)')
axes[2].set_ylabel('Maximum Sustained Wind (kts)')
axes[2].legend(loc='upper left', fontsize=9)
axes[2].grid(True, linestyle=':', alpha=0.5)

# Panel D: Quadrant Wind Radii Polar Plot
quadrants = ['NE', 'SE', 'SW', 'NW', 'NE']
theta = np.linspace(0, 2*np.pi, 5)
r34 = [145, 130, 95, 120, 145]
r50 = [80, 70, 50, 65, 80]
r64 = [40, 35, 25, 30, 40]
ax_polar = plt.subplot(1, 4, 4, projection='polar')
ax_polar.plot(theta, r34, label='34 kt (Gale)', color='#38bdf8', linewidth=2)
ax_polar.plot(theta, r50, label='50 kt (Storm)', color='#eab308', linewidth=2)
ax_polar.plot(theta, r64, label='64 kt (Hurricane)', color='#ef4444', linewidth=2)
ax_polar.set_xticks(theta[:-1])
ax_polar.set_xticklabels(quadrants[:-1])
ax_polar.set_title('Panel D: Quadrant Wind Radii (NM)', fontweight='bold')
ax_polar.legend(loc='lower right', bbox_to_anchor=(1.3, -0.1), fontsize=8)

plt.tight_layout()
plt.show()
